[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/solutions/part_05_classical_ml/ch26_pipelines_nested_cv/03_coding_exam_SOLUTION.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `ANNOTATED SOLUTION NOTEBOOK`


# Chapter 26 Part V Capstone Coding Exam — Production ML Pipelines & Nested CV (100 Points)

**Book Reference:** Chapter 26, Section 26.7 (Part V Capstone Coding Exam)  
**Suggested Time:** 50–60 minutes (Open Book, Closed Solution)

| Task | Production ML Workflow Deliverable | Points |
| :--- | :--- | :---: |
| **Task 26.1** | `build_dhs_end_to_end_pipeline(num_cols, cat_cols, model)` (Sentinel 999 + mixed `ColumnTransformer` + `Pipeline`) | 30 |
| **Task 26.2** | `run_nested_cv_audit(pipe, param_grid, X, y, outer_splits=4, inner_splits=3, seed=42)` (True Nested Stratified CV) | 35 |
| **Task 26.3** | `benchmark_part5_capstone(mch_df, seed=42)` (Cluster-aware `GroupKFold` 3-model leaderboard on `child_stunted`) | 35 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Part V Capstone Exam Grader
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import (
    GridSearchCV,
    GroupKFold,
    StratifiedKFold,
    cross_val_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import sharukh_tutor as tutor


class SentinelToNaNTransformer(BaseEstimator, TransformerMixin):
    """Provided helper from To-Do 26.1 for Task 26.1 & Task 26.3."""

    def __init__(self, sentinel=999):
        self.sentinel = sentinel

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            return X.copy().replace(self.sentinel, np.nan)
        arr = np.asarray(X, dtype=float).copy()
        arr[arr == self.sentinel] = np.nan
        return arr


tutor.activate(chapter="ch26", notebook="coding_exam")


---
## Exam Task 26.1 (30 Points): `build_dhs_end_to_end_pipeline(num_cols, cat_cols, model)`

Return a single leak-free `Pipeline([("prep", prep), ("model", model)])` whose `"prep"` `ColumnTransformer` chains:
1. On `num_cols`: `SentinelToNaNTransformer(999)` $\rightarrow$ `SimpleImputer(strategy="median")` $\rightarrow$ `StandardScaler()`.
2. On `cat_cols`: `SimpleImputer(strategy="most_frequent")` $\rightarrow$ `OneHotEncoder(handle_unknown="ignore", sparse_output=False)`.

Your pipeline must handle sentinel `999`, `NaN`, and unseen test categories seamlessly without crashing.

In [ ]:
def build_dhs_end_to_end_pipeline(num_cols, cat_cols, model):
    """Return leak-free DHS Pipeline with sentinel-999 & mixed prep."""
    num_pipe = Pipeline([
        ("sentinel", SentinelToNaNTransformer(999)),
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore", sparse_output=False
        )),
    ])
    prep = ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ])
    return Pipeline([("prep", prep), ("model", model)])


---
## Exam Task 26.2 (35 Points): `run_nested_cv_audit(pipe, param_grid, X, y, outer_splits=4, inner_splits=3, seed=42)`

Implement true **Nested Stratified Cross-Validation** to eliminate hyperparameter-selection optimism bias:
1. Outer loop: `StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=seed)`.
2. Inside each outer training fold, run `GridSearchCV(pipe, param_grid, cv=StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=seed), scoring="roc_auc")`.
3. Score `gs.best_estimator_` on the untouched outer test fold with `roc_auc_score` (rounded to 4 decimals) and record `gs.best_params_`.
4. Return `{"outer_mean_auc": round4, "outer_std_auc": round4, "outer_scores": list_round4, "fold_best_params": list_of_dicts}`.

In [ ]:
def run_nested_cv_audit(
    pipe, param_grid, X, y, outer_splits=4, inner_splits=3, seed=42
):
    """Run true Nested Stratified CV and return outer AUC audit."""
    outer_cv = StratifiedKFold(
        n_splits=outer_splits, shuffle=True, random_state=seed
    )
    outer_scores = []
    fold_best_params = []
    for tr_idx, te_idx in outer_cv.split(X, y):
        X_tr = X.iloc[tr_idx] if hasattr(X, "iloc") else X[tr_idx]
        X_te = X.iloc[te_idx] if hasattr(X, "iloc") else X[te_idx]
        y_tr = y.iloc[tr_idx] if hasattr(y, "iloc") else y[tr_idx]
        y_te = y.iloc[te_idx] if hasattr(y, "iloc") else y[te_idx]
        inner_cv = StratifiedKFold(
            n_splits=inner_splits, shuffle=True, random_state=seed
        )
        gs = GridSearchCV(
            pipe, param_grid, cv=inner_cv, scoring="roc_auc"
        )
        gs.fit(X_tr, y_tr)
        probs = gs.best_estimator_.predict_proba(X_te)[:, 1]
        outer_scores.append(round(float(roc_auc_score(y_te, probs)), 4))
        fold_best_params.append(gs.best_params_)
    return {
        "outer_mean_auc": round(float(np.mean(outer_scores)), 4),
        "outer_std_auc": round(float(np.std(outer_scores)), 4),
        "outer_scores": outer_scores,
        "fold_best_params": fold_best_params,
    }


---
## Exam Task 26.3 (35 Points): `benchmark_part5_capstone(mch_df, seed=42)`

Without mutating `mch_df`, run the Part V Capstone model benchmark predicting `child_stunted`:
1. Copy `mch_df` and clean `residence` casing via `.astype(str).str.strip().str.title()`.
2. Use `num_cols = ["maternal_age_years", "anc_visits", "maternal_hb_g_dl"]` and `cat_cols = ["residence", "wealth_quintile"]`.
3. Wrap each candidate model in `build_dhs_end_to_end_pipeline(num_cols, cat_cols, model)`:
   - `"LogisticRegression"`: `LogisticRegression(max_iter=500)`
   - `"RandomForestClassifier"`: `RandomForestClassifier(n_estimators=80, max_depth=6, random_state=seed)`
   - `"HistGradientBoostingClassifier"`: `HistGradientBoostingClassifier(max_depth=4, random_state=seed)`
4. Score all three with `GroupKFold(n_splits=5)` grouped by `cluster_id` and `scoring="roc_auc"`.
5. Return `{"leaderboard": df_sorted, "winner": top_model_name}` where `leaderboard` has columns `["model", "mean_auc", "std_auc"]` (rounded to 4 decimals), sorted descending by `mean_auc` with a clean index.

In [ ]:
def benchmark_part5_capstone(mch_df, seed=42):
    """Benchmark 3 leak-free DHS pipelines under 5-fold GroupKFold."""
    work = mch_df.copy()
    work["residence"] = (
        work["residence"].astype(str).str.strip().str.title()
    )
    num_cols = [
        "maternal_age_years", "anc_visits", "maternal_hb_g_dl"
    ]
    cat_cols = ["residence", "wealth_quintile"]
    X = work[num_cols + cat_cols]
    y = work["child_stunted"]
    groups = work["cluster_id"]
    gkf = GroupKFold(n_splits=5)

    models = {
        "LogisticRegression": LogisticRegression(max_iter=500),
        "RandomForestClassifier": RandomForestClassifier(
            n_estimators=80, max_depth=6, random_state=seed
        ),
        "HistGradientBoostingClassifier": HistGradientBoostingClassifier(
            max_depth=4, random_state=seed
        ),
    }
    rows = []
    for name, mdl in models.items():
        pipe = build_dhs_end_to_end_pipeline(num_cols, cat_cols, mdl)
        scores = cross_val_score(
            pipe, X, y, groups=groups, cv=gkf, scoring="roc_auc"
        )
        rows.append({
            "model": name,
            "mean_auc": round(float(scores.mean()), 4),
            "std_auc": round(float(scores.std()), 4),
        })
    leaderboard = (
        pd.DataFrame(rows)
        .sort_values("mean_auc", ascending=False)
        .reset_index(drop=True)
    )
    return {
        "leaderboard": leaderboard,
        "winner": str(leaderboard.loc[0, "model"]),
    }


In [ ]:
exam_submissions = {
    "ch26_exam_01": build_dhs_end_to_end_pipeline,
    "ch26_exam_02": run_nested_cv_audit,
    "ch26_exam_03": benchmark_part5_capstone,
}

tutor.check_exam("ch26", exam_submissions)


---
## Part V Capstone Exam Debugging Post-Mortem (Solution Notebook Only)

- **Task 26.1 (Sentinel Ordering Inside `ColumnTransformer`):** `SentinelToNaNTransformer(999)` MUST sit **before** `SimpleImputer(strategy="median")`. If you place it after imputation (or omit it), `999` inflates the training median and standard deviation and causes rows with missing `anc_visits=999` to receive absurd $z$-scores.
- **Task 26.2 (Why Nested CV Beats Flat `GridSearchCV`):** Reporting `gs.best_score_` from a single CV loop uses the same folds to both select hyperparameters and estimate generalization—creating optimism bias. Nested CV keeps the outer test fold completely invisible to inner hyperparameter selection.
- **Task 26.3 (Simple Regularized Models Often Win on Small-N Surveys):** Under honest cluster-level `GroupKFold` validation on `mch_survey_sample` ($n = 500$, 50 clusters, 70 stunting cases), `LogisticRegression` (`0.7168`) beats both `RandomForestClassifier` (`0.6870`) and `HistGradientBoostingClassifier` (`0.6665`). Always benchmark linear baselines fairly before deploying complex ensembles!